In [ ]:
import pandas as pd
import joblib

df_model = pd.read_csv('../data/processed/processed_logs.csv')
feature_columns = joblib.load('../trained_models/feature_columns.pkl')

X = df_model[feature_columns]
y = df_model['label']

In [ ]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.1, random_state=42)
iso.fit(X)

df_model['anomaly_score'] = iso.decision_function(X)
df_model['is_anomaly'] = iso.predict(X)

In [ ]:
print(X.dtypes)
print(X.select_dtypes(include='object').columns.tolist())

In [ ]:
print(pd.crosstab(df_model['is_anomaly'], df_model['label']))

In [ ]:
actual_attack_rate = df_model['label'].value_counts(normalize=True)['Attack']
print(actual_attack_rate)  # should be ~0.05

iso = IsolationForest(contamination=actual_attack_rate, random_state=42)
iso.fit(X)
df_model['is_anomaly'] = iso.predict(X)

print(pd.crosstab(df_model['is_anomaly'], df_model['label']))

In [ ]:
import joblib

joblib.dump(iso, '../trained_models/isolation_forest.pkl')

import os
print(os.path.exists('../trained_models/isolation_forest.pkl'))  # should print True